In [9]:
print("Hello World!")

Hello World!


In [10]:
import requests
import pandas as pd
from defs import defs

In [11]:
session = requests.Session()

In [12]:
ins_df = pd.read_pickle("instruments.pkl")

In [13]:
our_curr = ['EUR', 'USD', 'GBP', 'JPY', 'CHF', 'NZD', 'CAD']

In [14]:
# ins_df.shape # num rows, num cols
# ins_df.info()
# ins_df.columns
# ins_df.name.unique()

In [15]:
def fetch_candles(pair_name, count, granularity):
    url = f"{defs.OANDA_URL}/instruments/{pair_name}/candles"
    params = dict(
        count = count,
        granularity = granularity,
        price = "MBA"
    )
    response = session.get(url, params=params, headers=defs.SECURE_HEADER)
    return response.status_code, response.json()


In [16]:
code, res = fetch_candles("EUR_USD", 10, "H1")

In [17]:
print(code)

200


In [18]:
print(res)

{'instrument': 'EUR_USD', 'granularity': 'H1', 'candles': [{'complete': True, 'volume': 4690, 'time': '2026-05-08T11:00:00.000000000Z', 'bid': {'o': '1.17634', 'h': '1.17681', 'l': '1.17613', 'c': '1.17645'}, 'mid': {'o': '1.17641', 'h': '1.17690', 'l': '1.17620', 'c': '1.17652'}, 'ask': {'o': '1.17648', 'h': '1.17698', 'l': '1.17628', 'c': '1.17660'}}, {'complete': True, 'volume': 12867, 'time': '2026-05-08T12:00:00.000000000Z', 'bid': {'o': '1.17648', 'h': '1.17781', 'l': '1.17545', 'c': '1.17733'}, 'mid': {'o': '1.17655', 'h': '1.17788', 'l': '1.17576', 'c': '1.17741'}, 'ask': {'o': '1.17662', 'h': '1.17795', 'l': '1.17588', 'c': '1.17749'}}, {'complete': True, 'volume': 11482, 'time': '2026-05-08T13:00:00.000000000Z', 'bid': {'o': '1.17732', 'h': '1.17865', 'l': '1.17655', 'c': '1.17780'}, 'mid': {'o': '1.17740', 'h': '1.17872', 'l': '1.17664', 'c': '1.17789'}, 'ask': {'o': '1.17749', 'h': '1.17880', 'l': '1.17673', 'c': '1.17798'}}, {'complete': True, 'volume': 10318, 'time': '202

In [19]:
def get_candles_df(json_response):

    prices = ['mid', 'bid', 'ask']
    ohlc = ['o', 'h', 'l', 'c']
    
    our_data = []
    for candle in json_response['candles']:
        if candle['complete'] == False:
            continue
        new_dict = {}
        new_dict['time'] = candle['time']
        new_dict['volume'] = candle['volume']
        for price in prices:
            for oh in ohlc:
                new_dict[f"{price}_{oh}"] = candle[price][oh]
        our_data.append(new_dict)
    return pd.DataFrame.from_dict(our_data)

In [20]:
code, res = fetch_candles("EUR_USD", 10, "H1")

In [21]:
df = get_candles_df(res)

In [22]:
df.head()

,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c
0,2026-05-08T11:00:00.000000000Z,4690,1.17641,1.17690,1.17620,1.17652,1.17634,1.17681,1.17613,1.17645,1.17648,1.17698,1.17628,1.17660
1,2026-05-08T12:00:00.000000000Z,12867,1.17655,1.17788,1.17576,1.17741,1.17648,1.17781,1.17545,1.17733,1.17662,1.17795,1.17588,1.17749
2,2026-05-08T13:00:00.000000000Z,11482,1.17740,1.17872,1.17664,1.17789,1.17732,1.17865,1.17655,1.17780,1.17749,1.17880,1.17673,1.17798
3,2026-05-08T14:00:00.000000000Z,10318,1.17791,1.17797,1.17673,1.17744,1.17784,1.17789,1.17665,1.17737,1.17798,1.17805,1.17681,1.17752
4,2026-05-08T15:00:00.000000000Z,8553,1.17745,1.17786,1.17670,1.17728,1.17737,1.17777,1.17661,1.17720,1.17753,1.17794,1.17678,1.17737


In [23]:
def save_file(candles_df, pair, granularity):
    candles_df.to_pickle(f"his_data/{pair}_{granularity}.pkl")

In [24]:
def create_data(pair, granularity):
    code, json_data = fetch_candles(pair, 4000, granularity)
    if code != 200:
        print(pair, "Error")
        return
    df = get_candles_df(json_data)
    print(f"{pair} loaded {df.shape[0]} candles from {df.time.min()} to {df.time.max()}")
    save_file(df, pair, granularity)


In [25]:
for p1 in our_curr:
    for p2 in our_curr:
        pair = f"{p1}_{p2}"
        if pair in ins_df.name.unique():
            create_data(pair, "H1")

EUR_USD loaded 4000 candles from 2025-09-16T05:00:00.000000000Z to 2026-05-08T20:00:00.000000000Z
EUR_GBP loaded 4000 candles from 2025-09-16T05:00:00.000000000Z to 2026-05-08T20:00:00.000000000Z
EUR_JPY loaded 4000 candles from 2025-09-16T05:00:00.000000000Z to 2026-05-08T20:00:00.000000000Z
EUR_CHF loaded 4000 candles from 2025-09-16T05:00:00.000000000Z to 2026-05-08T20:00:00.000000000Z
EUR_NZD loaded 4000 candles from 2025-09-16T05:00:00.000000000Z to 2026-05-08T20:00:00.000000000Z
EUR_CAD loaded 4000 candles from 2025-09-16T05:00:00.000000000Z to 2026-05-08T20:00:00.000000000Z
USD_JPY loaded 4000 candles from 2025-09-16T05:00:00.000000000Z to 2026-05-08T20:00:00.000000000Z
USD_CHF loaded 4000 candles from 2025-09-16T05:00:00.000000000Z to 2026-05-08T20:00:00.000000000Z
USD_CAD loaded 4000 candles from 2025-09-16T05:00:00.000000000Z to 2026-05-08T20:00:00.000000000Z
GBP_USD loaded 4000 candles from 2025-09-16T05:00:00.000000000Z to 2026-05-08T20:00:00.000000000Z
GBP_JPY loaded 4000 